# 04 · Evaluate the audio transformer on the test set

Runs the test recordings through every final model of `02_train_audio_transformer`. Nothing is fitted or chosen here:

1. **Test similarities** of all six subjects for every setting (test EEG band-passed and normalised with the statistics of the training recordings; sound features of the 102 test videos, scaled with the training videos).
2. **Test scores** (top-1, top-5, mean rank) for sub-01 to sub-04, whose test labels exist, next to their validation scores from `02`.
3. **Submission** for sub-05 and sub-06 from the setting that `02` chose on validation (`models/chosen.json`), plus sub-01 with the same code to check the format. The test scores of the other settings are reported for completeness; they do not change the choice.
4. **Failure cases** of the chosen model: the top-k curve, whether the same videos are hard for every subject, and the videos without sound.

**Saved** next to the evaluated models: `audio_transformer/<run of 02>/test/` (tables, figures, similarity matrices, `submission/`).

In [ ]:
from math import gcd
from pathlib import Path
import json, math, os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm

## 1. Configuration

In [ ]:
DATA_ROOT = Path(os.environ.get('EEG_DATA_ROOT', str(Path.home() / 'eeg-data' / 'eeg-mind-reading-challenge'))).expanduser()
if (DATA_ROOT / 'eeg-mind-reading-challenge').is_dir():
    DATA_ROOT /= 'eeg-mind-reading-challenge'
RESULTS_ROOT = Path(os.environ.get('EEG_RESULTS_ROOT', str(Path.home() / 'eeg-results'))).expanduser()
OUT_ROOT = RESULTS_ROOT / 'pretrained-transformers' / 'audio_transformer'
TRAIN_RUN = None                     # run of 02 to evaluate (e.g. '2026-10-19_101500'); None = the latest finished one

SUBJECTS = [1, 2, 3, 4, 5, 6]
DEV_SUBJECTS = [1, 2, 3, 4]          # test labels exist (tst_stim): scored here
FINAL_SUBJECTS = [5, 6]              # labels withheld: the submission
TEST_VIDEO_IDS = np.arange(1001, 1103)
N_CANDIDATES = len(TEST_VIDEO_IDS)
DEVICE = 'cuda:0'                    # inference only: one GPU is enough

finished = sorted(p.parent.parent for p in OUT_ROOT.glob('*/models/chosen.json'))
assert TRAIN_RUN or finished, f'no finished run of 02 in {OUT_ROOT}'
TRAIN_DIR = OUT_ROOT / TRAIN_RUN if TRAIN_RUN else finished[-1]
TEST_DIR = TRAIN_DIR / 'test'
for d in [TEST_DIR / 'similarity', TEST_DIR / 'submission', TEST_DIR / 'figures']:
    d.mkdir(parents=True, exist_ok=True)
CONFIG = json.loads((TRAIN_DIR / 'config.json').read_text())
CHOSEN = json.loads((TRAIN_DIR / 'models' / 'chosen.json').read_text())
MODELS = [json.loads(p.read_text()) for p in sorted((TRAIN_DIR / 'models').glob('*.json')) if p.name != 'chosen.json']
print(f'run {TRAIN_DIR.name}: {len(MODELS)} models', [m['name'] for m in MODELS], '\nchosen on validation:', CHOSEN['name'])

## 2. Model and preprocessing (as in `02`)

The same dual encoder and EEG preprocessing as in `02`, with the sizes read from its `config.json`. The test EEG keeps its original row order; it is normalised with each subject's **training** statistics (mean response, channel scale), and the sound features with the scaling of the training videos, both stored with the models.

In [ ]:
ARCH = CONFIG['architecture']
(N_FEATURES, N_SLOTS, N_CHANNELS, N_STEPS, HIDDEN, LATENT, KERNEL, DILATIONS, INPUT_DROPOUT, DROPOUT, GROUP_NORM_GROUPS,
 EEG_KERNEL, TF_DIM, TF_LAYERS, TF_HEADS, TF_DROPOUT, BAND_HZ, RATE_HZ, EEG_SFREQ) = (ARCH[k] for k in [
    'n_features', 'n_slots', 'n_channels', 'n_steps', 'hidden', 'latent', 'kernel', 'dilations', 'input_dropout',
    'dropout', 'group_norm_groups', 'eeg_kernel', 'tf_dim', 'tf_layers', 'tf_heads', 'tf_dropout', 'band_hz', 'rate_hz',
    'eeg_sfreq'])
OUR_SLOT = {int(k): v for k, v in ARCH['our_slot'].items()}


class CausalBlock(nn.Module):
    def __init__(self, dilation):
        super().__init__()
        self.pad = (KERNEL - 1) * dilation
        self.conv = nn.Conv1d(HIDDEN, 2 * HIDDEN, KERNEL, dilation=dilation)
        self.norm = nn.GroupNorm(GROUP_NORM_GROUPS, HIDDEN)
        self.drop = nn.Dropout(DROPOUT)

    def forward(self, x):                                # (B, HIDDEN, T)
        y = F.glu(self.conv(F.pad(x, (self.pad, 0))), dim=1)
        return x + self.drop(self.norm(y))


class DualEncoder(nn.Module):
    """As in 02; the initial values do not matter here, all weights are loaded."""

    def __init__(self, n_features, n_slots, eeg_encoder):
        super().__init__()
        self.stim_in = nn.Sequential(nn.Dropout(INPUT_DROPOUT), nn.Linear(n_features, HIDDEN))
        self.stim_blocks = nn.Sequential(*[CausalBlock(d) for d in DILATIONS])
        self.stim_out = nn.Conv1d(HIDDEN, LATENT, 1)
        self.subject_proj = nn.Parameter(torch.zeros(n_slots, LATENT, N_CHANNELS))
        self.shared_proj = nn.Linear(N_CHANNELS, LATENT, bias=False)
        self.eeg_conv = nn.Conv1d(LATENT, LATENT, EEG_KERNEL, padding=EEG_KERNEL // 2, groups=LATENT)
        self.use_tf = eeg_encoder == 'cortiva+transformer'
        if self.use_tf:
            self.tf_in = nn.Linear(LATENT, TF_DIM)
            self.tf_pos = nn.Parameter(torch.zeros(1, N_STEPS, TF_DIM))
            layer = nn.TransformerEncoderLayer(TF_DIM, TF_HEADS, 2 * TF_DIM, TF_DROPOUT, activation='gelu',
                                               batch_first=True, norm_first=True)
            self.tf = nn.TransformerEncoder(layer, TF_LAYERS, enable_nested_tensor=False)
            self.tf_out = nn.Linear(TF_DIM, LATENT)
        self.logit_scale = nn.Parameter(torch.zeros(()))

    @staticmethod
    def zscore(z):                                       # per latent channel over time, then flattened
        z = (z - z.mean(-1, keepdim=True)) / (z.std(-1, keepdim=True) + 1e-6)
        return z.flatten(1) / math.sqrt(z.shape[1] * z.shape[2])

    def stimulus(self, feats):                           # (B, 150, K) -> (B, LATENT * 150)
        h = self.stim_in(feats).transpose(1, 2)
        return self.zscore(self.stim_out(self.stim_blocks(h)))

    def eeg(self, x, slot):                              # (B, 17, 150) -> (B, LATENT * 150)
        w = self.shared_proj.weight + self.subject_proj[slot]
        z = self.eeg_conv(torch.bmm(w, x))
        if self.use_tf:
            h = self.tf(self.tf_in(z.transpose(1, 2)) + self.tf_pos)
            z = z + self.tf_out(h).transpose(1, 2)
        return self.zscore(z)


@torch.no_grad()
def scores(model, X, slot, feats):
    """Similarity of EEG windows X (n, 17, 150) to candidate videos feats (m, 150, K); candidates centred."""
    model.eval()
    s = model.stimulus(feats)
    return model.eeg(X, slot) @ (s - s.mean(0, keepdim=True)).T


def bandpass_resample(x, sfreq, device=DEVICE):
    """As in 02: FFT band-pass to BAND_HZ and resampling to RATE_HZ in one step, each row on its own."""
    step = sfreq // gcd(sfreq, RATE_HZ)
    x = torch.as_tensor(np.asarray(x)[:, :x.shape[1] // step * step], dtype=torch.float64, device=device)
    n = x.shape[1]
    m = n * RATE_HZ // sfreq
    spec = torch.fft.rfft(torch.cat([x, x.flip(-1)], dim=-1), dim=-1, norm='forward')
    f = torch.fft.rfftfreq(2 * n, d=1 / sfreq, dtype=torch.float64, device=device)
    spec[..., (f < BAND_HZ[0]) | (f > BAND_HZ[1])] = 0
    return torch.fft.irfft(spec[..., :m + 1], n=2 * m, dim=-1, norm='forward')[..., :m].float().cpu().numpy()


def load_test_eeg(sub):
    """Test recordings (102, 17, 150) of one subject, in the original order of tst/sub-XX_tst.npy."""
    eeg = np.load(DATA_ROOT / 'tst' / f'sub-{sub:02d}_tst.npy')
    n, c, _ = eeg.shape
    return bandpass_resample(eeg.reshape(n * c, -1), EEG_SFREQ).reshape(n, c, -1)


def row_z(S):
    return (S - S.mean(axis=1, keepdims=True)) / S.std(axis=1, keepdims=True)


def true_ranks(S, true_cols):
    """Rank of the true video (1 = first) in the ranked list a submission contains (stable sort). Silent videos get
    identical scores; this way such ties are never broken in the model's favour."""
    order = np.argsort(-np.asarray(S), axis=1, kind='stable')
    return np.argmax(order == np.asarray(true_cols)[:, None], axis=1) + 1


def metrics(S, true_cols):
    rank = true_ranks(S, true_cols)
    return dict(n=len(rank), top1=float(np.mean(rank == 1)), top5=float(np.mean(rank <= 5)), mean_rank=float(rank.mean()))


TEST_EEG = {sub: load_test_eeg(sub) for sub in tqdm(SUBJECTS, desc='Loading test EEG', unit='subject')}
LABELS = {sub: np.load(DATA_ROOT / 'tst_stim' / f'sub-{sub:02d}_tst_stim.npy').reshape(-1) for sub in DEV_SUBJECTS}
assert all(np.array_equal(np.sort(v), TEST_VIDEO_IDS) for v in LABELS.values())
TRUE_COLS = {sub: np.searchsorted(TEST_VIDEO_IDS, v) for sub, v in LABELS.items()}

VID_FEATS = np.load(TRAIN_DIR / 'features' / 'video_sound_features.npy')     # (1102, 150, K), rows = videos 1..1102
SILENT = np.load(TRAIN_DIR / 'features' / 'video_is_silent.npy')
assert VID_FEATS.shape == (1102, N_STEPS, N_FEATURES)
TEST_ROWS = TEST_VIDEO_IDS - 1
print(f'{SILENT[TEST_ROWS].sum()} of the {N_CANDIDATES} test videos are silent:', TEST_VIDEO_IDS[SILENT[TEST_ROWS]].tolist())

## 3. Test similarities of every model

Per setting: each of the 4 seeds scores the test recordings against the 102 test videos; each seed's scores are row-z-scored and averaged (the seed ensemble of `02`). Saved per setting and subject: rows = test recordings in their original order, columns = videos 1001–1102.

In [ ]:
SIMS = {}
for cfg in tqdm(MODELS, desc='Test similarities', unit='model'):
    saved = torch.load(TRAIN_DIR / 'models' / f"{cfg['name']}.pt", map_location=DEVICE, weights_only=True)
    models = []
    for seed in cfg['seeds']:
        model = DualEncoder(N_FEATURES, N_SLOTS, cfg['encoder']).to(DEVICE)
        model.load_state_dict(saved['states'][seed])
        models.append(model)
    fn = saved['feature_normalisation']
    cand = (torch.as_tensor(VID_FEATS[TEST_ROWS], device=DEVICE) - fn['mean']) / fn['std']
    (TEST_DIR / 'similarity' / cfg['name']).mkdir(exist_ok=True)
    SIMS[cfg['name']] = {}
    for sub in SUBJECTS:
        en = saved['eeg_normalisation'][sub]
        x = ((torch.as_tensor(TEST_EEG[sub], device=DEVICE) - en['mean']) / en['scale']).float()
        slot = torch.full((len(x),), OUR_SLOT[sub], device=DEVICE)
        S = np.mean([row_z(scores(m, x, slot, cand.float()).cpu().numpy()) for m in models], axis=0)
        SIMS[cfg['name']][sub] = S
        np.save(TEST_DIR / 'similarity' / cfg['name'] / f'sub-{sub:02d}_tst_similarity.npy', S)

## 4. Test scores (sub-01 to sub-04) next to validation

Chance: top-1 0.010, top-5 0.049. Validation = 5 folds of 102 held-out **training** videos (`02`), test = the 102 test videos. If the ranking of the settings agrees, the validation protocol was a sound basis for the choice.

In [ ]:
TEST = pd.DataFrame([dict(model=cfg['name'], subject=sub, **metrics(SIMS[cfg['name']][sub], TRUE_COLS[sub]))
                     for cfg in MODELS for sub in DEV_SUBJECTS])
VAL = pd.DataFrame([dict(model=c['name'], val_top1=c['validation']['top1'], val_top5=c['validation']['top5'])
                    for c in MODELS]).set_index('model')
SUMMARY = (TEST.groupby('model')[['top1', 'top5', 'mean_rank']].mean().join(VAL)
           .sort_values('val_top5', ascending=False))
SUMMARY['chosen'] = SUMMARY.index == CHOSEN['name']
TEST.to_csv(TEST_DIR / 'test_per_subject.csv', index=False)
SUMMARY.reset_index().to_csv(TEST_DIR / 'test_summary.csv', index=False)

x = np.arange(len(SUMMARY))
per_sub = TEST.pivot(index='model', columns='subject', values='top5').loc[SUMMARY.index]
fig, ax = plt.subplots(figsize=(8, 4.2))
ax.bar(x - 0.2, SUMMARY.val_top5, 0.4, color='0.7', label='validation (held-out training videos)')
ax.bar(x + 0.2, SUMMARY.top5, 0.4, color='C0', label='test (sub-01 to sub-04)')
for i, vals in enumerate(per_sub.values):
    ax.plot(np.full(len(vals), i + 0.2), vals, 'k.', ms=3)
ax.axhline(5 / N_CANDIDATES, color='0.4', ls='--', lw=1, label='chance')
ax.set_xticks(x)
ax.set_xticklabels([n + ('  [chosen]' if c else '') for n, c in zip(SUMMARY.index, SUMMARY.chosen)],
                   rotation=30, ha='right', fontsize=8)
ax.set_ylabel('top-5')
ax.set_title('Audio transformer: validation vs test (dots = test subjects)', fontsize=10)
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(TEST_DIR / 'figures' / 'validation_vs_test.png', dpi=130, bbox_inches='tight')
plt.show()
SUMMARY.round(3)

## 5. Submission (setting chosen on validation in `02`)

`sub-XX_ranked_candidate_list.npy`: 102 × 102 integers. Row *i* belongs to the *i*-th recording of `tst/sub-XX_tst.npy` in its original order and holds all video IDs 1001–1102, from most to least likely. sub-01 is written with the same code and scored from the file itself, to check the format.

In [ ]:
for sub in FINAL_SUBJECTS + [DEV_SUBJECTS[0]]:
    ranked = TEST_VIDEO_IDS[np.argsort(-SIMS[CHOSEN['name']][sub], axis=1, kind='stable')].astype(int)
    assert ranked.shape == (102, 102) and all(np.array_equal(np.sort(r), TEST_VIDEO_IDS) for r in ranked)
    np.save(TEST_DIR / 'submission' / f'sub-{sub:02d}_ranked_candidate_list.npy', ranked)

check = np.load(TEST_DIR / 'submission' / 'sub-01_ranked_candidate_list.npy')
place = np.array([np.flatnonzero(row == v)[0] for row, v in zip(check, LABELS[1])])       # 0 = first place
m = metrics(SIMS[CHOSEN['name']][1], TRUE_COLS[1])
print(f"format check, sub-01 scored from its file: top-1 {np.mean(place == 0):.3f}, top-5 {np.mean(place < 5):.3f} "
      f"(from the similarities: {m['top1']:.3f}, {m['top5']:.3f}; must be equal)")
print('written:', sorted(p.name for p in (TEST_DIR / 'submission').glob('*.npy')))

## 6. Failure cases of the chosen model

- **Top-k curve.** The share of recordings whose true video is among the first *k* places, for every *k*. Chance is the diagonal *k*/102.
- **Same videos hard for every subject?** If the failures were only single-trial noise, the rank of a video in one subject would say nothing about another subject (Spearman *r* ≈ 0; with 102 videos, |*r*| > 0.2 is unlikely by chance, p < 0.05). A positive *r* means that some sounds are systematically hard, e.g. steady sounds without onsets, which evoke little time-locked response.
- **Silent videos.** Without sound, all silent videos get the same features, so the audio model cannot tell them apart and can only place them as a group. A clear failure by design: here only the visual model (`03`) can help.

In [ ]:
S_CHOSEN = SIMS[CHOSEN['name']]
RANK = pd.DataFrame({f'sub-{sub:02d}': pd.Series(true_ranks(S_CHOSEN[sub], TRUE_COLS[sub]), index=LABELS[sub])
                     for sub in DEV_SUBJECTS}).sort_index()                   # rows: test videos, values: rank
RANK['mean'] = RANK.mean(axis=1)
RANK['silent'] = SILENT[RANK.index.values - 1]
RANK.rename_axis('video').reset_index().to_csv(TEST_DIR / 'failure_cases_rank_per_video.csv', index=False)
subject_cols = [f'sub-{sub:02d}' for sub in DEV_SUBJECTS]

k = np.arange(1, N_CANDIDATES + 1)
fig, ax = plt.subplots(figsize=(6, 3.8))
for col in subject_cols:
    ax.plot(k, [(RANK[col] <= kk).mean() for kk in k], lw=1.2, label=col)
ax.plot(k, k / N_CANDIDATES, 'k--', lw=1, label='chance')
ax.set_xscale('log')
ax.set_xlabel('k (log scale)')
ax.set_ylabel('share of recordings with the true video in the top k')
ax.set_title(f"Top-k curve: {CHOSEN['name']}", fontsize=10)
ax.legend(fontsize=8)
fig.savefig(TEST_DIR / 'figures' / 'failure_topk_curve.png', dpi=130, bbox_inches='tight')
plt.show()

r = RANK[subject_cols].corr(method='spearman').values[np.triu_indices(len(subject_cols), 1)]
print(f'rank of the same video across subjects: mean pairwise Spearman r = {r.mean():.2f} '
      f'(pairs from {r.min():.2f} to {r.max():.2f})')
print('easiest videos (mean rank over sub-01..04):', RANK['mean'].nsmallest(5).round(1).to_dict())
print('hardest videos:', RANK['mean'].nlargest(5).round(1).to_dict())
for silent, group in RANK.groupby('silent'):
    ranks_g = group[subject_cols].values
    print(f"{'silent' if silent else 'with sound'}: {len(group)} videos, mean rank {ranks_g.mean():.1f}, "
          f'top-5 {np.mean(ranks_g <= 5):.3f}')

**Reference result** (same pipeline in `pretrained_transformers/06`, 2026-10-18; chosen: `scratch`, `cortiva+transformer`): test top-1 0.351, top-5 0.606 (sub-01: 0.461 / 0.696, sub-02: 0.245 / 0.549, sub-03: 0.461 / 0.608, sub-04: 0.235 / 0.569); an earlier run with `scratch`, `cortiva` gave 0.363 / 0.596. Chance 0.010 / 0.049. Those runs broke ties between silent videos in the model's favour, so the stable-sort scores here can be slightly lower. Sound alone thus identifies about a third of the test recordings exactly, ~3.4× the top-1 of the video transformer in `03` (0.103).